# Imports and Config

In [ ]:
import os
import pickle
from grf_pipeline_utils.signal_processing import *
from grf_pipeline_utils.opensim_utils import *
import yaml

repo_root = os.path.abspath('../')
with open(os.path.join(repo_root, 'config.yaml')) as f:
    cfg = yaml.safe_load(f)

root_dir        = os.path.join(repo_root, cfg['silder']['data_root'])
transformed_dir = os.path.join(repo_root, cfg['silder']['results']['transformed'])
processed_dir   = os.path.join(repo_root, cfg['silder']['results']['processed'])
scaling_dir     = os.path.join(repo_root, cfg['silder']['results']['scaling'])
ik_dir          = os.path.join(repo_root, cfg['silder']['results']['ik'])
id_dir          = os.path.join(repo_root, cfg['silder']['results']['id_raw'])

OA_subjects  = [f"OA{i}" for i in cfg['silder']['OA_subjects']]
Y_subjects   = [f"Y{i}"  for i in cfg['silder']['Y_subjects']]

speeds       = cfg['silder']['speeds']
n_trials     = cfg['silder']['n_trials']
subj_masses  = {**cfg['silder']['OA_masses'], **cfg['silder']['Y_masses']}

# Build Subject_Trials Dictionary

In [2]:
subjects = OA_subjects + Y_subjects
trial_names = []

subject_trials = {}
for subj in subjects:
    subj_dir = os.path.join(root_dir, subj, 'Walking/Files_W_HJCs/')
    # OA subjects use '_walk_static1' while Y subjects use '_walking_static1'
    static_name = f'{subj}_walk_static1.trc' if subj[0] == 'O' else f'{subj}_walking_static1.trc'
    subject_trials[subj] = {
        'static': {
            'input':  os.path.join(subj_dir, static_name),
            'output': os.path.join(transformed_dir, f'{subj}_walk_static1_transformed.trc')
        },
        'tracking': [],
        'forces': []
    }
    for spd in speeds:
        for i in range(1, n_trials + 1):
            trial_name = f'{subj}_{spd}_{i}'
            trial_names.append(trial_name)
            subject_trials[subj]['tracking'].append({
                'input':  os.path.join(subj_dir, f'{trial_name}.trc'),
                'output': os.path.join(transformed_dir, f'{trial_name}_transformed.trc')
            })
            subject_trials[subj]['forces'].append({
                'input':  os.path.join(subj_dir, f'{trial_name}.forces'),
                'output': os.path.join(transformed_dir, f'{trial_name}_transformed.mot')
            })

# Preprocess Tracking and GRF Files

In [4]:
all_segs = {}
for subj, data in subject_trials.items():
    process_hjc_trc(input_path=data['static']['input'],
                    output_path=data['static']['output'],
                    markers_to_drop=[])
    for trc, forces in zip(data['tracking'], data['forces']):
        trial_segs = preprocess_trc_grf(
            trc_ip=trc['input'],
            trc_op=trc['output'],
            markers_to_drop=[],
            grf_ip=forces['input'],
            grf_op=forces['output'],
            grf_pickle_path=os.path.join(processed_dir, 'grf_pickles')
        )
        for trial_name, seg_dict in trial_segs.items():
            subj_name = trial_name.split('_')[0]
            if subj_name not in all_segs:
                all_segs[subj_name] = {}
            all_segs[subj_name][trial_name] = seg_dict

with open(os.path.join(processed_dir, 'all_stance_segs.pkl'), 'wb') as f:
    pickle.dump(all_segs, f)
print('Saved all_segs.pkl')

one peak found!, trial: OA1_100_4, start time: 1101
one peak found!, trial: OA4_120_1, start time: 2868
one peak found!, trial: OA5_100_3, start time: 1210
one peak found!, trial: OA9_120_5, start time: 566
one peak found!, trial: OA17_120_4, start time: 1762
one peak found!, trial: OA18_120_2, start time: 399
one peak found!, trial: OA18_120_3, start time: 508
one peak found!, trial: OA18_120_4, start time: 629
one peak found!, trial: OA18_120_5, start time: 546
one peak found!, trial: OA19_80_4, start time: 305
one peak found!, trial: OA19_100_1, start time: 789
one peak found!, trial: OA19_100_4, start time: 799
one peak found!, trial: OA19_100_5, start time: 661
one peak found!, trial: OA19_120_2, start time: 954
one peak found!, trial: OA19_120_4, start time: 1555
one peak found!, trial: OA25_120_2, start time: 2403
one peak found!, trial: Y1_100_2, start time: 1842
one peak found!, trial: Y1_120_3, start time: 1073
one peak found!, trial: Y5_120_3, start time: 338
one peak found!

# Scaling

In [ ]:
for subj, data in subject_trials.items():
    scale_generic(
        root_dir=root_dir,
        mass=subj_masses[subj],
        static_pose_filename=data['static']['output'],
        scaling_dir=scaling_dir
    )

# Parse Scaling Log

In [6]:
scaling_log = os.path.join(scaling_dir, 'Combined_scaling_log.txt')
parsed = parse_combined_scaling_output(scaling_log)
for subject, info in parsed.items():
    print(f"{subject} — RMS: {info['marker_error_rms']:.4f}, "
          f"max: {info['marker_error_max']:.4f} at {info['marker_error_max_marker']}")

Read scaling log
OA1 — RMS: 0.0174, max: 0.0287 at R.ASIS
OA2 — RMS: 0.0158, max: 0.0314 at L.Knee
OA4 — RMS: 0.0119, max: 0.0195 at R.Heel
OA5 — RMS: 0.0158, max: 0.0248 at L.Heel
OA7 — RMS: 0.0137, max: 0.0251 at L.Knee
OA8 — RMS: 0.0171, max: 0.0315 at R.Knee
OA9 — RMS: 0.0129, max: 0.0308 at L.Knee
OA10 — RMS: 0.0156, max: 0.0268 at L.Knee
OA11 — RMS: 0.0162, max: 0.0319 at L.Heel
OA12 — RMS: 0.0145, max: 0.0328 at L.Knee
OA13 — RMS: 0.0171, max: 0.0319 at L.Heel
OA14 — RMS: 0.0222, max: 0.0360 at R.Heel
OA17 — RMS: 0.0215, max: 0.0454 at R.Knee
OA18 — RMS: 0.0141, max: 0.0251 at R.Knee
OA19 — RMS: 0.0203, max: 0.0380 at L.Knee
OA20 — RMS: 0.0139, max: 0.0279 at L.Knee
OA22 — RMS: 0.0155, max: 0.0299 at L.Heel
OA24 — RMS: 0.0148, max: 0.0241 at L.Knee
OA25 — RMS: 0.0188, max: 0.0290 at L.Knee
Y1 — RMS: 0.0209, max: 0.0354 at R.ASIS
Y2 — RMS: 0.0178, max: 0.0309 at L.ASIS
Y4 — RMS: 0.0159, max: 0.0243 at R.Knee
Y5 — RMS: 0.0191, max: 0.0353 at R.Knee
Y6 — RMS: 0.0132, max: 0.0221 at

Inverse kinematics too memory-intensive to run in notebook

# Parse IK Log

In [4]:
ik_log = os.path.join(ik_dir, 'Combined_ik_log.txt')
ik_df = parse_full_ik_log(ik_log, trial_names)
problem_trials = []
mean_rms_count, mean_max_count = 0, 0

for idx, row in ik_df.iterrows():
    if row['mean_rms'] > 0.04:
        mean_rms_count += 1
        problem_trials.append(row['trial_name'])
    elif row['mean_max'] > 0.05:
        mean_max_count += 1
        problem_trials.append(row['trial_name'])

print(f'{mean_rms_count} trials with Mean RMS > 4 cm')
print(f'{mean_max_count} trials with Mean max error > 5 cm')
for t in problem_trials:
    print(t)

Read inverse kinematics log
24 trials with Mean RMS > 4 cm
15 trials with Mean max error > 5 cm
OA5_80_1
OA5_80_2
OA5_80_3
OA5_80_4
OA5_100_4
OA5_100_5
OA5_120_1
OA5_120_4
OA5_120_5
OA11_80_1
OA11_80_2
OA11_80_3
OA11_80_4
OA11_80_5
OA11_100_3
OA11_100_4
OA11_100_5
OA11_120_1
OA11_120_2
OA11_120_3
OA11_120_4
OA11_120_5
OA12_80_2
OA17_80_1
Y2_80_1
Y2_80_4
Y2_100_3
Y2_120_1
Y2_120_2
Y2_120_4
Y2_120_5
Y5_100_3
Y10_80_5
Y10_100_1
Y10_100_3
Y12_80_3
Y12_100_3
Y15_120_3
Y17_80_4


# Inverse Dynamics

In [4]:
loads_dir       = os.path.join(repo_root, cfg['silder']['results']['loads'])
id_raw_dir      = os.path.join(repo_root, cfg['silder']['results']['id_raw'])
id_filtered_dir = os.path.join(repo_root, cfg['silder']['results']['id_filtered'])

for subj, data in subject_trials.items():
    model = osim.Model(os.path.join(scaling_dir, f'{subj}_scaled.osim'))
    for trc, forces in zip(data['tracking'], data['forces']):
        inverse_dynamics(
            root_dir=root_dir,
            force_data_filepath=forces['output'],
            tracking_data_filepath=trc['output'],
            model=model,
            loads_dir=loads_dir,
            id_raw_dir=id_raw_dir,
            id_filtered_dir=id_filtered_dir,
            setup_dir = os.path.join(repo_root, cfg['silder']['opensim_setup_dir'])        )

[info] Loaded model OA1_scaled from file /Users/briankeller/Desktop/GRFMusclePrediction/results/Silder/Scaling/OA1_scaled.osim
[warning] Couldn't find file 'r_pelvis.vtp'.
[warning] Couldn't find file 'l_pelvis.vtp'.
[warning] Couldn't find file 'sacrum.vtp'.
[warning] Couldn't find file 'r_femur.vtp'.
[warning] Couldn't find file 'r_tibia.vtp'.
[warning] Couldn't find file 'r_fibula.vtp'.
[warning] Couldn't find file 'r_patella.vtp'.
[warning] Couldn't find file 'r_talus.vtp'.
[warning] Couldn't find file 'r_foot.vtp'.
[warning] Couldn't find file 'r_bofoot.vtp'.
[warning] Couldn't find file 'l_femur.vtp'.
[warning] Couldn't find file 'l_tibia.vtp'.
[warning] Couldn't find file 'l_fibula.vtp'.
[warning] Couldn't find file 'l_patella.vtp'.
[warning] Couldn't find file 'l_talus.vtp'.
[warning] Couldn't find file 'l_foot.vtp'.
[warning] Couldn't find file 'l_bofoot.vtp'.
[warning] Couldn't find file 'hat_spine.vtp'.
[warning] Couldn't find file 'hat_jaw.vtp'.
[warning] Couldn't find file